# 10-714 家庭作业 2

在本次作业中，您将在 needle 框架中实现一个神经网络库。提醒 您必须在 `drive__` 中保存一份副本。

In [ ]:
# Code to set up the assignment
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/
!mkdir -p 10714
%cd /content/drive/MyDrive/10714
!git clone https://github.com/dlsys10714/hw2.git
%cd /content/drive/MyDrive/10714/hw2

!pip3 install --upgrade --no-deps git+https://github.com/dlsys10714/mugrade.git

### Setting some variables

In [ ]:
MY_API_KEY = "<FILL YOUR API KEY HERE>"
HW2_NAME = "hw2"

## 问题 0

本作业建立在作业 1 的基础上。

**首先，在作业 2 目录下，复制作业 1 中的文件 `python/needle/autograd.py`、`python/needle/ops/ops_mathematic.py`。

***注意****： 张量的默认数据类型是 `float32`。如果想更改数据类型，可以在 `Tensor` 构造函数中设置 `dtype` 参数。例如，`Tensor([1, 2, 3], dtype='float64')` 将创建一个数据类型为`float64`的张量。
在这项工作中，**确保您创建的任何张量都是 `float32` 数据类型，以避免自动跟踪器出现任何问题**。

In [ ]:
import sys
sys.path.append('./python')
sys.path.append('./apps')

## 问题 1

在第一个问题中，你将实现几种不同的权重初始化方法。这将在 `python/needle/init/init_initializers.py` 文件中完成，该文件包含一系列使用各种随机和常量初始化方法来初始化 needle 张量的例程。遵循现有初始化器的相同方法（你需要在下面的函数中调用例如 `init.rand` 或 `init.randn`，这些已在 `python/needle/init/init_basic.py` 中实现），实现以下常见的初始化方法。在所有情况下，函数应返回 `fan_in` 乘以 `fan_out` 的二维张量（可通过例如重塑扩展到其他尺寸）。

### Xavier 均匀初始化
`xavier_uniform(fan_in, fan_out, gain=1.0, **kwargs)`

根据 [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf) 中描述的方法，使用均匀分布填充输入张量。生成的张量将从 $\mathcal{U}(-a, a)$ 中采样值，其中

$$a = \text{gain} \times \sqrt{\frac{6}{\text{fan\_in} + \text{fan\_out}}}$$

将剩余的 `**kwargs` 参数传递给相应的 `init` 随机调用。

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `gain` - 可选缩放因子
___

### Xavier 正态初始化
`xavier_normal(fan_in, fan_out, gain=1.0, **kwargs)`

根据 [Understanding the difficulty of training deep feedforward neural networks](https://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf) 中描述的方法，使用正态分布填充输入张量。生成的张量将从 $\mathcal{N}(0, \text{std}^2)$ 中采样值，其中

$$\mathrm{std} = \mathrm{gain} \times \sqrt{\frac{2}{\mathrm{fan}_{in} + \mathrm{fan}_{out}}}$$

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `gain` - 可选缩放因子
___

### Kaiming 均匀初始化
`kaiming_uniform(fan_in, fan_out, nonlinearity="relu", **kwargs)`

根据 [Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification](https://arxiv.org/pdf/1502.01852.pdf) 中描述的方法，使用均匀分布填充输入张量。生成的张量将从 $\mathcal{U}(-\text{bound}, \text{bound})$ 中采样值，其中

$$\mathrm{bound} = \mathrm{gain} \times \sqrt{\frac{3}{\mathrm{fan}_{in}}}$$

对于 ReLU，使用推荐的增益值：$\text{gain}=\sqrt{2}$。

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `nonlinearity` - 非线性函数
___

### Kaiming 正态初始化
`kaiming_normal(fan_in, fan_out, nonlinearity="relu", **kwargs)`

根据 [Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification](https://arxiv.org/pdf/1502.01852.pdf) 中描述的方法，使用正态分布填充输入张量。生成的张量将从 $\mathcal{N}(0, \text{std}^2)$ 中采样值，其中

$$\mathrm{std} = \frac{\mathrm{gain}}{\sqrt{\mathrm{fan}_{in}}}$$

对于 ReLU，使用推荐的增益值：$\text{gain}=\sqrt{2}$。

##### 参数
- `fan_in` - 输入维度
- `fan_out` - 输出维度
- `nonlinearity` - 非线性函数

In [ ]:
!python3 -m pytest -v -k "test_init"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "init"

## 问题 2

在这个问题中，你将在 `python/needle/nn/nn_basic.py` 中实现额外的模块。具体来说，对于下面描述的以下模块，在构造函数中初始化模块的任何变量，并填写 `forward` 方法。**注意：** 确保使用你刚刚实现的 `init` 函数来初始化参数，并且不要忘记传递 `dtype` 参数。
___

### 线性层
`needle.nn.Linear(in_features, out_features, bias=True, device=None, dtype="float32")`

对输入数据应用线性变换：$y = xA^T + b$。输入形状为 $(N, H_{\text{in}})$，其中 $H_{\text{in}}=\text{in\_features}$。输出形状为 $(N, H_{\text{out}})$，其中 $H_{\text{out}}=\text{out\_features}$。

**注意显式将偏置项广播到正确的形状——Needle 不支持隐式广播。**

**注意：对于包括此层在内的所有层，你应先初始化权重张量，再初始化偏置张量，并且应仅使用 `init` 中的函数初始化所有参数**。存在此初始化顺序要求是因为 mugrade 上的测试用例答案是在假设权重在偏置参数之前初始化的前提下准备的。虽然先初始化偏置再初始化权重在算法上仍然是正确的，但模型解决方案和预期的测试输出是使用权重优先的约定生成的。因此，为了通过 mugrade 上的自动化测试，你必须遵循此特定的初始化顺序。如果你在本作业的其他部分对哪个层或参数应首先初始化有任何疑问，请在 Ed 上提出请求以获取澄清。

##### 参数
- `in_features` - 每个输入样本的大小
- `out_features` - 每个输出样本的大小
- `bias` - 如果设置为 `False`，该层将不学习加性偏置。

##### 变量
- `weight` - 可学习的权重，形状为 (`in_features`, `out_features`)。值应使用 **Kaiming 均匀初始化** 进行初始化，其中 `fan_in = in_features`
- `bias` - 可学习的偏置，形状为 (`out_features`)。值应使用 Kaiming 均匀初始化进行初始化，其中 `fan_in = out_features`。**注意由于它们的相对大小不同，fan_in 的选择也不同**。

**注意：** 确保将所有必要的变量（例如 `weight`、`bias`）封装在 **`Parameter` 类** 中，以便它们对接下来将实现的优化器可见。**你可以阅读 `python/needle/nn/nn_basic.py` 中的 `class Parameter` 和函数 `unpack_params` 来了解更多信息。**

In [ ]:
!python3 -m pytest -v -k "test_nn_linear"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_linear"

### ReLU
`needle.nn.ReLU()`

逐元素应用修正线性单元函数：
$ReLU(x) = max(0, x)$。

如果你之前基于 ReLU 自身实现过其反向传播，请注意这在数值上是不稳定的，可能会在后续过程中引发问题。

相反，考虑到我们可以将 ReLU 的导数写为 $I\{x>0\}$，其中，在本作业中，我们任意决定并固定一个约定：将 $x=0$ 处的导数视为 0。
（这是一个*次可微*函数。）

___

In [ ]:
!python3 -m pytest -v -k "test_nn_relu"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_relu"

### 顺序容器
`needle.nn.Sequential(*modules)`

将一系列模块按顺序（按照它们传递给构造函数的顺序）应用于输入，并返回最后一个模块的输出。
这些模块应保存在 `.modules` 属性中：你**不应**重新定义任何魔术方法如 `__getitem__`，因为这可能与我们的测试不兼容。

##### 参数
- `*modules` - 任意数量的 `needle.nn.Module` 类型模块

___

In [ ]:
!python3 -m pytest -v -k "test_nn_sequential"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_sequential"

### LogSumExp

`needle.ops.LogSumExp(axes)`

对输入应用数值稳定的 log-sum-exp 函数，通过减去最大元素来实现。你需要在文件 `python/needle/ops/ops_logarithmic.py` 中实现此操作及下一个操作。

\begin{equation}
\text{LogSumExp}(z) = \log (\sum_{i} \exp (z_i - \max{z})) + \max{z}
\end{equation}

#### 参数
- `axes` - 要求和并取最大元素的轴元组。这使用与 `needle.ops.Summation()` 相同的约定

##### 为什么采用这种形式？处理数值稳定性

- **朴素定义：**
  LogSumExp 最直接的定义是

  $$
  \log \left(\sum_i \exp(z_i)\right)
  $$

- **问题：**
  这种朴素计算容易产生数值不稳定性。
  - 如果某些 $z_i$ 非常大（例如 $1000$），则 $\exp(z_i)$ 会溢出为 $\infty$。
  - 如果某些 $z_i$ 非常小（例如 $-1000$），则 $\exp(z_i)$ 会下溢为 $0$。
  这两种情况在浮点运算中都会扭曲结果。

- **修复方法：**
  为避免此问题，我们提取最大元素 $M = \max(z)$：

  $$
  \log \left(\sum_i \exp(z_i)\right)
  = \log \left(\exp(M)\sum_i \exp(z_i - M)\right)
  = M + \log \left(\sum_i \exp(z_i - M)\right).
  $$

  现在所有指数项最多为 $\exp(0) = 1$，因此完全避免了溢出。

- **下溢呢？**
  如果 $z_i - M$ 非常负（例如 $-1000$），下溢仍可能发生，因为在浮点中 $\exp(-1000) \approx 0$。
  然而，这不是问题：这样的项与最大值相比已经可以忽略不计，不会对求和产生有意义的影响。

以下博客文章也是一个很好的参考：https://indii.org/blog/gradients-of-softmax-and-logsumexp/
___

In [ ]:
!python3 -m pytest -v -k "test_op_logsumexp"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "op_logsumexp"

### LogSoftmax

`needle.ops.LogSoftmax(axes)`

对输入应用数值稳定的 logsoftmax 函数，通过减去最大元素来实现。
对于此问题，你可以假设输入 NDArray 是二维的，并且我们在 `axis=1` 上执行 softmax。

\begin{equation}
\text{LogSoftmax}(z) = \log \left(\frac{\exp(z_i - \max z)}{\sum_{i}\exp(z_i - \max z)}\right) = z - \text{LogSumExp}(z)
\end{equation}
___

In [ ]:
!python3 -m pytest -v -k "test_op_logsoftmax"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "op_logsoftmax"

### SoftmaxLoss

`needle.nn.SoftmaxLoss()` 位于 `python/needle/nn/nn_basic.py`

应用如下定义的 softmax 损失（与作业 1 中的实现相同），输入为一个 logits 张量和一个真实标签张量（表示为数字列表，**非** one-hot 编码）。

注意，你现在可以使用 `init.one_hot` 函数，而无需自己编写。注意：为此你需要使用刚刚实现的数值稳定的 logsumexp 操作符。

\begin{equation}
\ell_\text{softmax}(z,y) = \log \sum_{i=1}^k \exp z_i - z_y
\end{equation}

___

In [ ]:
!python3 -m pytest -v -k "test_nn_softmax_loss"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_softmax_loss"

### LayerNorm1d

`needle.nn.LayerNorm1d(dim, eps=1e-5, device=None, dtype="float32")`

对输入的小批量应用层归一化，如论文 [Layer Normalization](https://arxiv.org/abs/1607.06450) 中所述。

\begin{equation}
y = w \circ \frac{x_i - \textbf{E}[x]}{((\textbf{Var}[x]+\epsilon)^{1/2})} + b
\end{equation}

其中 $\textbf{E}[x]$ 表示输入的经验均值，$\textbf{Var}[x]$ 表示其经验方差（注意这里我们使用方差的"有偏"估计，即除以 $N$ 而非 $N-1$），而 $w$ 和 $b$ 分别表示可学习的标量权重和偏置。注意，你可以假设该层的输入是一个二维张量，第一维为批次，第二维为特征。在应用权重和偏置之前，你可能需要进行广播。

##### 参数
- `dim` - 通道数
- `eps` - 添加到分母中的值，用于数值稳定性

##### 变量
- `weight` - 可学习的权重，大小为 `dim`，元素初始化为 1
- `bias` - 可学习的偏置，形状为 `dim`，元素初始化为 0
___

In [ ]:
!python3 -m pytest -v -k "test_nn_layernorm"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_layernorm"

### Flatten
`needle.nn.Flatten()`

接收形状为 `(B, X_0, X_1, ...)` 的张量，并展平所有非批次维度，使得输出形状为 `(B, X_0 * X_1 * ...)`。

In [ ]:
!python3 -m pytest -v -k "test_nn_flatten"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_flatten"

### BatchNorm1d

`needle.nn.BatchNorm1d(dim, eps=1e-5, momentum=0.1, device=None, dtype="float32")`

对输入的小批量应用批归一化，如论文 [Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift](https://arxiv.org/abs/1502.03167) 中所述。

\begin{equation}
y = w \circ \frac{z_i - \textbf{E}[x]}{((\textbf{Var}[x]+\epsilon)^{1/2})} + b
\end{equation}

但这里的均值和方差指的是在**批次维度**上的均值和方差。该函数还会计算每层所有特征的均值和方差的滑动平均值 $\hat{\mu}, \hat{\sigma}^2$，并在测试时使用这些值进行归一化：

\begin{equation}
y = \frac{(x - \hat{\mu})}{((\hat{\sigma}^2_{i+1})_j+\epsilon)^{1/2}}
\end{equation}

BatchNorm 在测试时使用均值和方差的滑动估计值而非批次统计量，即在 BatchNorm 层的 `training` 标志设为 false 后（例如调用 `model.eval()` 后）。

要计算滑动估计值，你可以使用公式 $$\hat{x_{new}} = (1 - m) \hat{x_{old}} + mx_{observed},$$
其中 $m$ 是动量参数。

##### 参数
- `dim` - 输入维度
- `eps` - 添加到分母中的值，用于数值稳定性
- `momentum` - 用于计算滑动均值和滑动方差的动量值

##### 变量
- `weight` - 可学习的权重，大小为 `dim`，元素初始化为 1
- `bias` - 可学习的偏置，大小为 `dim`，元素初始化为 0
- `running_mean` - 评估时使用的滑动均值，元素初始化为 0
- `running_var` - 评估时使用的滑动（无偏）方差，元素初始化为 1

___

In [ ]:
!python3 -m pytest -v -k "test_nn_batchnorm"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_batchnorm"

### Dropout

`needle.nn.Dropout(p = 0.5)`

在训练期间，使用伯努利分布的样本以概率 `p` 随机将输入张量的部分元素置零。这已被证明是一种有效的正则化技术，可以防止神经元的共适应，如论文 [Improving neural networks by preventing co-adaptation of feature detectors](https://arxiv.org/abs/1207.0580) 中所述。在评估期间，该模块仅执行恒等函数。

\begin{equation}
\hat{z}_{i+1} = \sigma_i (W_i^T z_i + b_i) \\
(z_{i+1})_j = 
    \begin{cases}
    (\hat{z}_{i+1})_j /(1-p) & \text{以概率 } 1-p \\
    0 & \text{以概率 } p \\
    \end{cases}
\end{equation}

除以 \(1-p\) 是为了保持期望激活值不变，因为

$$
\mathbb{E}\big[\operatorname{Dropout}((\hat{z}_{i+1})_j)\big]
= (1-p)\,\frac{(\hat{z}_{i+1})_j}{1-p} + p \cdot 0
= (\hat{z}_{i+1})_j
$$

**重要提示**：如果 Dropout 模块的标志 `training=False`，则不应"丢弃"任何激活值。也就是说，dropout 仅在训练期间应用，在评估期间不应用。注意 `training` 是 `nn.Module` 中的一个标志。

##### 参数
- `p` - 元素被置零的概率

在实现 Dropout 时，`python/needle/init/init_basic.py` 中的工具函数可能会有所帮助。
___

In [ ]:
!python3 -m pytest -v -k "test_nn_dropout"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_dropout"

### Residual

`needle.nn.Residual(fn: Module)`

对给定模块 $\mathcal{F}$ 和输入张量 $x$ 应用残差或跳跃连接，返回 $\mathcal{F}(x) + x$。

##### 参数
- `fn` - 类型为 `needle.nn.Module` 的模块

In [ ]:
!python3 -m pytest -v -k "test_nn_residual"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "nn_residual"

### SGD

`needle.optim.SGD(params, lr=0.01, momentum=0.0, weight_decay=0.0)`

实现随机梯度下降（可选择性地使用动量，如下式中的 $\beta$ 所示）。

\begin{equation}
\begin{split}
    u_{t+1} &= \beta u_t + (1-\beta) \nabla_\theta f(\theta_t) \\
    \theta_{t+1} &= \theta_t - \alpha u_{t+1}
\end{split}
\end{equation}

##### 参数
- `params` - 要优化的参数的可迭代对象，类型为 `needle.nn.Parameter`
- `lr` (*float*) - 学习率
- `momentum` (*float*) - 动量因子
- `weight_decay` (*float*) - 权重衰减（L2 惩罚）

本作业中可以跳过 `clip_grad_norm` 的实现。
___

In [ ]:
!python3 -m pytest -v -k "test_optim_sgd"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "optim_sgd"

### Adam

`needle.optim.Adam(params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0)`

实现 Adam 算法，该算法在论文 [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980) 中提出。

\begin{equation}
\begin{split}
u_{t+1} &= \beta_1 u_t + (1-\beta_1) \nabla_\theta f(\theta_t) \\
v_{t+1} &= \beta_2 v_t + (1-\beta_2) (\nabla_\theta f(\theta_t))^2 \\
\hat{u}_{t+1} &= u_{t+1} / (1 - \beta_1^t) \quad \text{(偏差校正)} \\
\hat{v}_{t+1} &= v_{t+1} / (1 - \beta_2^t) \quad \text{(偏差校正)}\\
\theta_{t+1} &= \theta_t - \alpha \hat{u_{t+1}}/(\hat{v}_{t+1}^{1/2}+\epsilon)
\end{split}
    \end{equation}

**重要提示：** 请注意是否应用了偏差校正。

##### 参数
- `params` - 要优化的参数的可迭代对象，类型为 `needle.nn.Parameter`
- `lr` (*float*) - 学习率
- `beta1` (*float*) - 用于计算梯度滑动平均的系数
- `beta2` (*float*) - 用于计算梯度平方滑动平均的系数
- `eps` (*float*) - 添加到分母以提高数值稳定性的项
- `weight_decay` (*float*) - 权重衰减（L2 惩罚）

**提示：** 为帮助处理内存问题，请尝试理解如何使用 `.data` 或 `.detach()`。

In [ ]:
!python3 -m pytest -v -k "test_optim_adam"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "optim_adam"

## 问题 4

在本问题中，你将实现两个数据原语：`needle.data.DataLoader` 和 `needle.data.Dataset`。`Dataset` 存储样本及其对应的标签，而 `DataLoader` 在 `Dataset` 周围包装一个可迭代对象，以便轻松访问样本。

对于本问题，你将在 `python/needle/data` 目录下工作。

### 变换

首先我们将实现一些在处理图像时有用的变换。目前我们将使用水平翻转和随机裁剪。请在 `needle/data/data_transforms.py` 中完成以下函数。
___ 

#### RandomFlipHorizontal
`needle.data.RandomFlipHorizontal(p = 0.5)`

以概率 `p` 水平翻转图像。

##### 参数
- `p` (*float*) - 翻转输入图像的概率。
___

#### RandomCrop
`needle.data.RandomCrop(padding=3)`

在图像的所有边添加填充，然后在随机位置将图像裁剪回原始大小。返回与原始图像大小相同的图像。

##### 参数
- `padding` (*int*) - 图像每个边界的填充量。

In [ ]:
!python3 -m pytest -v -k "flip_horizontal"
!python3 -m pytest -v -k "random_crop"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "flip_horizontal"
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "random_crop"

`Dataset` 负责定义你的数据内容（例如，图像/标签对）以及如何获取单个样本。`Dataloader` 负责控制数据如何输入到训练中（例如，批处理、打乱顺序、按周期迭代）。

### Dataset

`Dataset` 类的每个**子类**必须实现三个函数：`__init__`、`__len__` 和 `__getitem__`。
* `__init__` 函数初始化图像、标签和变换。
* `__len__` 函数返回数据集中的样本数量。
* `__getitem__` 函数从数据集中检索给定索引 `idx` 处的样本，**对图像调用变换函数（如果适用）**，将图像和标签转换为 numpy 数组（数据将在其他地方转换为张量）。`__getitem__` 和 `__next__` 的输出应为 NDArray，并且你应遵循形状规则，使得访问的数组大小为（数据点编号，特征维度 1，特征维度 2，...）。

请在 `needle/data/datasets/mnist_dataset.py` 中的 `MNISTDataset` 类中完成这些函数。你可以使用之前作业中 `parse_mnist` 的解决方案来实现 `__init__` 函数。

### MNISTDataset（Dataset 的子类）
`needle.data.MNISTDataset(image_filename, label_filename, transforms)`

##### 参数
- `image_filename` - 包含图像的文件路径
- `label_filename` - 包含标签的文件路径
- `transforms` - 要应用于数据的可选变换列表

In [ ]:
!python3 -m pytest -v -k "test_mnist_dataset"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "mnist_dataset"

### Dataloader

`needle.data.Dataloader(dataset: Dataset, batch_size: Optional[int] = 1, shuffle: bool = False)`

在 `needle/data/data_basic.py` 中，Dataloader 类提供了一个接口，用于组装适合基于 SGD 方法训练的小批量样本，由 `Dataset` 对象支持。为了构建典型的 Dataloader 接口（允许用户遍历数据集中的所有小批量），你需要在类中实现 `__iter__()` 和 `__next__()` 调用：
* `__iter__()` 在每个新周期开始时调用（即每当开始遍历 dataloader 时）。
* `__next__()` 每个批次调用一次，直到周期结束。

请注意，对 next 的后续调用需要你返回后续的批次，因此 next 不是一个纯函数。

##### 用途

组合数据集和采样器，并提供对给定数据集的可迭代访问。

##### 参数
- `dataset` - `needle.data.Dataset` - 数据集
- `batch_size` - `int` - 数据批处理的大小
- `shuffle` - `bool` - 设置为 ``True`` 以**在每个周期开始时**重新打乱数据，默认为 ``False``。
___

In [ ]:
!python3 -m pytest -v -k "test_dataloader"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "dataloader"

## 问题 5

既然你已经实现了我们神经网络库的所有必要组件，让我们构建并训练一个 MLP ResNet。对于本问题，你将在 `apps/mlp_resnet.py` 中工作。首先，按照以下描述完成 `ResidualBlock` 和 `MLPResNet` 函数：

### ResidualBlock
`ResidualBlock(dim, hidden_dim, norm=nn.BatchNorm1d, drop_prob=0.1)`

实现一个残差块，结构如下：

<p align="center">
  <img src="https://github.com/dlsyscourse/hw2/blob/f4c994506f2c76d7fdcc5a711a483e31b189afaa/figures/residualblock.png?raw=true" alt="Residual Block"/>
</p>

**注意**：如果图片无法显示，请查看 `figures` 目录中的图片。

其中第一个线性层的 `in_features=dim` 和 `out_features=hidden_dim`，最后一个线性层的 `out_features=dim`。返回类型为 `nn.Module` 的块。

##### 参数
- `dim` (*int*) - 输入维度
- `hidden_dim` (*int*) - 隐藏层维度
- `norm` (*nn.Module*) - 归一化方法
- `drop_prob` (*float*) - dropout 概率

___

### MLPResNet
`MLPResNet(dim, hidden_dim=100, num_blocks=3, num_classes=10, norm=nn.BatchNorm1d, drop_prob=0.1)`

实现一个 MLP ResNet，结构如下：

<p align="center">
  <img src="https://github.com/dlsyscourse/hw2/blob/f4c994506f2c76d7fdcc5a711a483e31b189afaa/figures/mlp_resnet.png?raw=true" alt="MLP Resnet"/>
</p>

其中第一个线性层的 `in_features=dim` 和 `out_features=hidden_dim`，每个 ResidualBlock 的 `dim=hidden_dim` 和 `hidden_dim=hidden_dim//2`。返回类型为 `nn.Module` 的网络。

##### 参数
- `dim` (*int*) - 输入维度
- `hidden_dim` (*int*) - 隐藏层维度
- `num_blocks` (*int*) - ResidualBlock 的数量
- `num_classes` (*int*) - 类别数量
- `norm` (*nn.Module*) - 归一化方法
- `drop_prob` (*float*) - dropout 概率 (0.1)

**注意**：模块的初始化顺序应与 Resnet 中的执行顺序匹配。
___ 

一旦你正确构建了深度学习模型架构，让我们使用新的神经网络库组件来训练网络。具体实现 `epoch` 和 `train_mnist` 函数。

### Epoch

`epoch(dataloader, model, opt=None)`

执行一个训练或评估周期，遍历整个训练数据集一次（类似于之前作业中的 `nn_epoch`）。返回平均错误率（*float* 类型）和所有样本的平均损失（*float* 类型）。如果给出了 `opt`，在函数开始时将模型设置为 `training` 模式；如果 `opt` 为 `None`，将模型设置为 `eval` 模式。设置模式时，使用 `.train()` 和 `.eval()` 而不是修改 training 属性。

##### 参数
- `dataloader` (*`needle.data.DataLoader`*) - 从训练数据集返回样本的数据加载器
- `model` (*`needle.nn.Module`*) - 神经网络
- `opt` (*`needle.optim.Optimizer`*) - 优化器实例，或 `None`

___

### Train Mnist

`train_mnist(batch_size=100, epochs=10, optimizer=ndl.optim.Adam, lr=0.001, weight_decay=0.001, hidden_dim=100, data_dir="data")`
              
为 MNIST 数据初始化一个训练数据加载器（`shuffle` 设置为 `True`）和一个测试数据加载器，并使用给定的优化器（如果 `opt` 不为 None）和 softmax 损失训练 `MLPResNet` 指定周期数。返回训练最后一个周期计算得到的训练错误率、训练损失、测试错误率、测试损失的元组。如果未指定任何参数，使用默认参数。

##### 参数
- `batch_size` (*int*) - 训练和测试数据加载器的批大小
- `epochs` (*int*) - 训练周期数
- `optimizer` (*`needle.optim.Optimizer` 类型*) - 使用的优化器类型
- `lr` (*float*) - 学习率
- `weight_decay` (*float*) - 权重衰减
- `hidden_dim` (*int*) - `MLPResNet` 的隐藏层维度
- `data_dir` (*str*) - 包含 MNIST 图像/标签文件的目录

In [ ]:
!python3 -m pytest -v -k "test_mlp"

In [ ]:
!python3 -m mugrade submit "$MY_API_KEY" "$HW2_NAME" -k "mlp_resnet"

我们鼓励你尝试使用 `mlp_resnet.py` 训练脚本进行实验。

你可以研究以下方面的影响：
- 在线性层上使用不同的初始化方法
- 增加 dropout 概率
- 添加变换（通过 Dataset 的 `transforms=` 关键字参数以列表形式传入），例如随机裁剪